# CUDA DPO Alignment — Veritas Verifier (FIXED)

Run this only after `cuda_qlora_artifacts.zip` or `checkpoints/cuda_qlora_verifier/` exists. This notebook verifies artifacts first, runs bounded DPO, and zips adapter/report outputs.


## 1. Runtime settings


In [ ]:
REPO_URL = "https://github.com/sushildalavi/veritas.git"
REPO_DIR = "Veritas"
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
QLORA_ADAPTER = "checkpoints/cuda_qlora_verifier"
DPO_OUTPUT_DIR = "checkpoints/cuda_dpo_verifier"
PREFERENCE_PAIRS = "data/processed/preference_pairs.jsonl"
REPORT_JSON = "reports/cuda_dpo_eval.json"
REPORT_MD = "reports/cuda_dpo_eval.md"
MAX_TRAIN_PAIRS = 0
MAX_EVAL_PAIRS = 0
MAX_EVAL_EXAMPLES = 0
EPOCHS = 2
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR = 2e-5
MAX_LENGTH = 384
MAX_PROMPT_LENGTH = 256
MAX_NEW_TOKENS = 160
SEED = 42


## 2. Check GPU


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("This notebook requires a CUDA GPU. Enable GPU runtime first.")


## 3. Install dependencies


In [ ]:
# Keep Colab/Kaggle's CUDA-matched torch stack intact. Do NOT upgrade torch here.
!pip install -q -U transformers peft bitsandbytes accelerate datasets trl scikit-learn sentencepiece safetensors


## 4. Clone / update repo


In [ ]:
import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull --ff-only || true
!pip install -q -e .


## 5. Add / verify QLoRA adapter artifacts


In [ ]:
from pathlib import Path
import shutil
import zipfile, glob

artifact_candidates = [
    Path("cuda_qlora_artifacts.zip"),
    Path("../cuda_qlora_artifacts.zip"),
    Path("/kaggle/working/cuda_qlora_artifacts.zip"),
]
artifact_zip = next((path for path in artifact_candidates if path.exists()), None)
adapter_dir = Path(QLORA_ADAPTER)

if artifact_zip is None:
    raise FileNotFoundError(
        "QLoRA artifact zip not found. Place cuda_qlora_artifacts.zip in the Colab/Kaggle working directory or its parent folder before running DPO."
    )

if artifact_zip.name == "cuda_qlora_artifacts.zip" and artifact_zip.parent != Path("."):
    print(f"Copying {artifact_zip} into the notebook working directory")
    shutil.copy2(artifact_zip, Path("cuda_qlora_artifacts.zip"))
    artifact_zip = Path("cuda_qlora_artifacts.zip")

if not adapter_dir.exists() and artifact_zip.exists():
    print("Extracting cuda_qlora_artifacts.zip")
    with zipfile.ZipFile(artifact_zip) as zf:
        zf.extractall(".")

required=[adapter_dir/"adapter_config.json", adapter_dir/"adapter_model.safetensors"]
for p in required:
    print(p, "=>", p.exists())
if not all(p.exists() for p in required):
    raise FileNotFoundError("QLoRA adapter missing. Upload cuda_qlora_artifacts.zip or copy checkpoints/cuda_qlora_verifier before running DPO.")
print("Adapter files:", glob.glob(str(adapter_dir/"*")))
print("Preference pairs exists:", Path(PREFERENCE_PAIRS).exists())


## 6. Write fixed DPO runner


In [ ]:
print("Use scripts/train_cuda_dpo.py in the repo; it is the canonical DPO runner for this notebook.")


## 7. Run DPO


In [ ]:
!python3 scripts/train_cuda_dpo.py \
    --base-model {BASE_MODEL} \
    --qlora-adapter-path {QLORA_ADAPTER} \
    --preference-pairs-file {PREFERENCE_PAIRS} \
    --output-dir {DPO_OUTPUT_DIR} \
    --report-json {REPORT_JSON} \
    --report-md {REPORT_MD} \
    --max-train-pairs {MAX_TRAIN_PAIRS} \
    --max-val-examples {MAX_EVAL_EXAMPLES} \
    --beta 0.05 \
    --max-length {MAX_LENGTH} \
    --max-prompt-length {MAX_PROMPT_LENGTH} \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --gradient-accumulation-steps {GRAD_ACCUM} \
    --learning-rate {LR} \
    --max-new-tokens {MAX_NEW_TOKENS}


## 8. Verify DPO artifacts


In [ ]:
import os, json, glob
paths=["checkpoints/cuda_dpo_verifier","checkpoints/cuda_dpo_verifier/adapter_config.json","checkpoints/cuda_dpo_verifier/adapter_model.safetensors","reports/cuda_dpo_eval.json","reports/cuda_dpo_eval.md"]
for p in paths:
    print(p, "=>", os.path.exists(p))
print("DPO checkpoint files:", glob.glob("checkpoints/cuda_dpo_verifier/*"))
with open("reports/cuda_dpo_eval.json") as f:
    print(json.dumps(json.load(f), indent=2))


## 9. Zip and download DPO artifacts


In [ ]:
!zip -r cuda_dpo_artifacts.zip     checkpoints/cuda_dpo_verifier     reports/cuda_dpo_eval.json     reports/cuda_dpo_eval.md


In [ ]:
try:
    from google.colab import files
    files.download("cuda_dpo_artifacts.zip")
except ImportError:
    print("On Kaggle, download cuda_dpo_artifacts.zip from the Output tab or /kaggle/working/Veritas.")


## Copy back into local repo

Copy these paths from the zip into your local repo:
- `checkpoints/cuda_dpo_verifier/`
- `reports/cuda_dpo_eval.json`
- `reports/cuda_dpo_eval.md`

Then CUDA DPO is complete and docs can be updated with real DPO metrics.
